In [3]:
import cv2
import datetime
import csv

# load cascade
face_cascade=cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

# load trained model
recognizer=cv2.face.LBPHFaceRecognizer_create()
recognizer.read("training.yml")


# Label map
label_map={
    0:"Kevin",
    1:'Martin'
}

marked=set()
import mysql.connector

# connect to mysql
conn=mysql.connector.connect(
    host="localhost",
    user="root",
    password="pass123",
    database="attendence_db"
)
cursor=conn.cursor()

def marked_attendence(name):
    import datetime
    now=datetime.datetime.now()
    date=now.strftime("%Y-%m-%d")
    time=now.strftime("%H:%M:%S")

    check_query="SELECT * FROM attendence WHERE name=%s and date=%s;"
    cursor.execute(check_query,(name,date))
    result=cursor.fetchone()
    if result is None:
        insert_query="INSERT INTO attendence (name,date,time) VALUES(%s,%s,%s)"
        cursor.execute(insert_query,(name,date,time))
        conn.commit()
cap=cv2.VideoCapture(0)



while True:
    ret,frame=cap.read()
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    faces=face_cascade.detectMultiScale(gray,scaleFactor=1.3,minNeighbors=5)
    for (x,y,w,h) in faces:
        face_roi=gray[y:y+h,x:x+w]
        face_roi=cv2.resize(face_roi,(200,200))
        label,confidence=recognizer.predict(face_roi)
        if confidence<100:
            name=label_map[label]
            marked_attendence(name)
            cv2.putText(frame,name,(x,y-10),cv2.FONT_HERSHEY_SIMPLEX,1,(123,0,255),2)
        else:
            name="Unknown"
            # cv2.putText(frame,name,(x,y-10),cv2.FONT_HERSHEY_SIMPLEX,1,(123,0,255),2)
        
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
    cv2.imshow("Attendence Marking System By Kevin Manjila",frame)
    if cv2.waitKey(1)& 0xFF==ord('q'):
        break
cap.release()
cursor.close()
conn.close()
cv2.destroyAllWindows()
